## Part 2 -- Agent & Demo

> Run **connection_and_data.ipynb** first to load data and store embeddings, then run this notebook to build and query the agent.

In [1]:
import os

# -- LLM Provider -------------------------------------------------------------
# Default: Ollama (local, free). To use OpenAI-compatible API:
#   set OLLAMA_BASE_URL=https://api.openai.com/v1
#   set OLLAMA_MODEL=gpt-4o
#   set OPENAI_API_KEY=sk-...
#   set EMBEDDING_MODEL=text-embedding-3-small
LLM_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
LLM_MODEL    = os.getenv("OLLAMA_MODEL", "qwen2.5")
LLM_API_KEY  = os.getenv("OPENAI_API_KEY", "ollama")  # ignored by Ollama

# -- Embedding model -----------------------------------------------------------
# Always use a dedicated embedding model -- not the chat model.
# nomic-embed-text is purpose-built for semantic similarity and runs locally via Ollama.
# For OpenAI: use text-embedding-3-small or text-embedding-3-large
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "nomic-embed-text")

# -- Oracle AI Database --------------------------------------------------------
DB_HOST     = os.getenv("DB_HOST", "localhost")
DB_PORT     = os.getenv("DB_PORT", "1521")
DB_SERVICE  = os.getenv("DB_SERVICE", "FREEPDB1")
DB_USER     = os.getenv("DB_USER", "system")
DB_PASSWORD = os.getenv("DB_PASSWORD", "Welcome1")
DB_DSN      = f"{DB_HOST}:{DB_PORT}/{DB_SERVICE}"

# -- Data generator ------------------------------------------------------------
DAYS_OF_HISTORY = int(os.getenv("DAYS_OF_HISTORY", "90"))
RANDOM_SEED     = 42

print(f"cridentials db:        {DB_USER} @ {DB_PASSWORD}")
print(f"LLM:        {LLM_MODEL} @ {LLM_BASE_URL}")
print(f"Embeddings: {EMBEDDING_MODEL}")
print(f"Oracle:     {DB_USER}@{DB_DSN}")
print(f"History:    {DAYS_OF_HISTORY} days")

cridentials db:        system @ Welcome1
LLM:        qwen2.5 @ http://host.docker.internal:11434
Embeddings: nomic-embed-text
Oracle:     system@oracle-db:1521/FREEPDB1
History:    90 days


## Imports

- **`oracledb`** -- Oracle's thin Python driver, no Oracle Client installation required
- **`langchain_ollama`** -- LLM and embedding wrappers for locally-running Ollama models
- **`langchain_oracledb`** -- Oracle-native vector store (`OracleVS`) and persistent chat history (`OracleChatMessageHistory`)
- **`langchain.agents`** -- ReAct agent that decides which tool to call at each reasoning step
- **`langchain.tools`** -- `@tool` decorator for wrapping Python functions as agent-callable tools

In [2]:
# Copyright (c) 2024 Oracle and/or its affiliates.
# Licensed under the Universal Permissive License v 1.0

import os
import uuid
import random
from datetime import datetime, timedelta

import oracledb
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_oracledb import OracleVS
from langchain_oracledb.vectorstores.oraclevs import DistanceStrategy
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain.agents import create_agent


## Connect to Oracle AI Database

Using `oracledb` in thin mode -- no Oracle Client installation needed. The connection
string follows the standard `host:port/service` format.

In [3]:
try:
    connection = oracledb.connect(user=DB_USER, password=DB_PASSWORD, dsn=DB_DSN)
    cursor = connection.cursor()
    cursor.execute("SELECT SYSDATE FROM DUAL")
    print(f"✅ Connected to Oracle AI Database: {cursor.fetchone()[0]}")
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("→ Check that docker compose up completed and Oracle shows as healthy")
    print("→ Run: docker ps | grep oracle-26ai")
    raise

# SYSTEM user defaults to the SYSTEM tablespace which uses manual segment space
# management (MSSM). Oracle JSON and VECTOR column types require automatic segment
# space management (ASSM). Switch the default tablespace to USERS which has ASSM.
cursor.execute("ALTER USER system DEFAULT TABLESPACE USERS QUOTA UNLIMITED ON USERS")
connection.commit()
print("✅ Default tablespace set to USERS (required for VECTOR/JSON columns)")


✅ Connected to Oracle AI Database: 2026-04-13 09:50:13
✅ Default tablespace set to USERS (required for VECTOR/JSON columns)


## Connect to Existing Vector Store

The embeddings were already created by **connection_and_data.ipynb**. Here we reconnect to the existing  table without re-embedding anything.

In [4]:
# -- Embedding model (must match the one used when creating the vectors) ------
try:
    embeddings = OllamaEmbeddings(
        model=EMBEDDING_MODEL,
        base_url=LLM_BASE_URL,
    )
    _ = embeddings.embed_query("connection test")
    print(f"✅ Embedding model ready: {EMBEDDING_MODEL} @ {LLM_BASE_URL}")
except Exception as e:
    print(f"❌ Embedding model unavailable: {e}")
    raise

# -- Reconnect to existing PIPELINE_LOG_VECTORS table (no re-embedding) -------
vector_store = OracleVS(
    client=connection,
    embedding_function=embeddings,
    table_name="PIPELINE_LOG_VECTORS",
    distance_strategy=DistanceStrategy.COSINE,
)
print("✅ Reconnected to PIPELINE_LOG_VECTORS vector store")


✅ Embedding model ready: nomic-embed-text @ http://host.docker.internal:11434
✅ Reconnected to PIPELINE_LOG_VECTORS vector store


---
## Step 3 -- Build the Agent

Setup is complete. Oracle now holds:
- **`PIPELINE_JOBS`** -- structured run metadata (DAG, task, status, duration, error codes)
- **`PIPELINE_LOGS`** -- free-text log lines linked to each job
- **`PIPELINE_LOG_VECTORS`** -- vector embeddings of every log message, stored natively in Oracle

The next cells build the reasoning agent that queries all three using natural language.

## LangChain Setup: LLM and Agent Tools

Three tools give the agent complementary access to the data:

| Tool | What it does | When the agent uses it |
|------|-------------|------------------------|
| `pipeline_sql_tool` | Executes read-only SQL against `PIPELINE_JOBS` | Counts, durations, error codes, date ranges |
| `log_search_tool` | Semantic search over `PIPELINE_LOG_VECTORS` | Questions about log text, errors, schema changes |
| `hybrid_pipeline_search` | SQL filter + `VECTOR_DISTANCE()` in one query | Questions that combine DAG/date filter with log content |

The `hybrid_pipeline_search` tool is the key differentiator -- it demonstrates Oracle's
converged query capability: structured and vector search in a single SQL statement,
with no application-level join or data movement.

In [5]:
# -- LLM -----------------------------------------------------------------------
try:
    llm = ChatOllama(model=LLM_MODEL, base_url=LLM_BASE_URL, temperature=0)
    _ = llm.invoke("Reply with one word: ready")
    print(f"\u2705 LLM ready: {LLM_MODEL} @ {LLM_BASE_URL}")
except Exception as e:
    print(f"\u274c LLM unavailable: {e}")
    print(f"\u2192 Ensure Ollama is running and the model is pulled:")
    print(f"    ollama pull {LLM_MODEL}")
    raise


# -- Tool 1: Structured SQL ----------------------------------------------------
@tool
def pipeline_sql_tool(sql: str) -> str:
    """
    Execute a read-only SQL SELECT query against the PIPELINE_JOBS table.
    Use this for structured questions: failure counts, durations, error codes,
    retry storms, date ranges, and per-DAG or per-task statistics.
    Returns results as a formatted string. Only SELECT statements are allowed.

    Available columns in PIPELINE_JOBS:
      job_id, dag_id, task_id, run_id, execution_date, start_time, end_time,
      duration_seconds, status, retry_count, rows_processed, error_code
    """
    print(f"[TOOL] pipeline_sql_tool called with: {sql[:80].strip()}")
    sql_upper = sql.strip().upper()
    if not sql_upper.startswith("SELECT"):
        return "ERROR: Only SELECT statements are permitted."
    for keyword in ("INSERT", "UPDATE", "DELETE", "DROP", "TRUNCATE", "ALTER"):
        if keyword in sql_upper:
            return f"ERROR: {keyword} statements are not permitted."
    try:
        cur = connection.cursor()
        cur.execute(sql)
        cols = [d[0] for d in cur.description]
        rows = cur.fetchall()
        if not rows:
            return "Query returned 0 rows."
        col_widths = [max(len(c), max((len(str(r[i])) for r in rows), default=0)) for i, c in enumerate(cols)]
        header = "  ".join(c.ljust(col_widths[i]) for i, c in enumerate(cols))
        sep    = "  ".join("-" * w for w in col_widths)
        lines  = [header, sep] + [
            "  ".join(str(v).ljust(col_widths[i]) for i, v in enumerate(row))
            for row in rows
        ]
        return f"{len(rows)} row(s):\n" + "\n".join(lines)
    except Exception as e:
        return f"SQL error: {e}"


# -- Tool 2: Semantic log search -----------------------------------------------
@tool
def log_search_tool(query: str) -> str:
    """
    Semantic similarity search over pipeline log messages.
    Use this for questions that require understanding log text: schema changes,
    error descriptions, data quality warnings, or silent issues that did not
    cause a job failure (status=success but log contains WARNING or INFO about issues).
    Returns the top 5 most semantically relevant log entries with job context.
    """
    print(f"[TOOL] log_search_tool called with: {query}")
    try:
        results = vector_store.similarity_search(query, k=5)
        if not results:
            return "No relevant log messages found."

        # Enrich with job metadata from PIPELINE_JOBS
        output_lines: list[str] = []
        for i, doc in enumerate(results, 1):
            log_id = doc.id if hasattr(doc, "id") else doc.metadata.get("id", "unknown")
            cur = connection.cursor()
            cur.execute(
                """
                SELECT j.dag_id, j.task_id, j.execution_date, j.status,
                       l.log_level, l.log_message
                FROM   PIPELINE_LOGS l
                JOIN   PIPELINE_JOBS j ON j.job_id = l.job_id
                WHERE  l.log_id = :1
                """,
                [log_id],
            )
            row = cur.fetchone()
            if row:
                dag_id, task_id, exec_date, status, log_level, log_message = row
                output_lines.append(
                    f"[{i}] {log_level} | {dag_id}.{task_id} | {exec_date} | status={status}\n"
                    f"    {log_message}"
                )
            else:
                output_lines.append(f"[{i}] {doc.page_content}")

        return "\n\n".join(output_lines)
    except Exception as e:
        return f"Search error: {e}"


# -- Tool 3: Hybrid structured + vector search ---------------------------------
@tool
def hybrid_pipeline_search(query: str, dag_id: str = "", date_from: str = "", date_to: str = "") -> str:
    """
    Combined structured filter + semantic vector search in a single Oracle SQL query.
    This is Oracle's converged query capability: filter by DAG and/or date range,
    then rank results by semantic similarity to the query text -- all in one statement.
    Use this when a question has both a structural constraint (specific DAG, time window)
    AND requires understanding log content.

    Parameters:
      query     : natural language description of what you're looking for (required)
      dag_id    : filter to a specific DAG name (optional, leave empty for all DAGs)
      date_from : start date filter in YYYY-MM-DD format (optional)
      date_to   : end date filter in YYYY-MM-DD format (optional)

    Returns top 10 log entries ranked by semantic relevance within the filter.
    """
    print(f"[TOOL] hybrid_pipeline_search called | query={query} | dag={dag_id} | from={date_from} | to={date_to}")
    try:
        query_embedding = embeddings.embed_query(query)

        where_clauses = []
        bind_params: list = [query_embedding]

        if dag_id:
            where_clauses.append("j.dag_id = :2")
            bind_params.append(dag_id)
        if date_from:
            bind_params.append(date_from)
            where_clauses.append(f"j.execution_date >= TO_DATE(:{len(bind_params)}, 'YYYY-MM-DD')")
        if date_to:
            bind_params.append(date_to)
            where_clauses.append(f"j.execution_date <= TO_DATE(:{len(bind_params)}, 'YYYY-MM-DD')")

        where_sql = ("WHERE " + " AND ".join(where_clauses)) if where_clauses else ""

        sql = f"""
            SELECT j.dag_id, j.task_id, j.execution_date, j.status,
                   l.log_level, l.log_message,
                   VECTOR_DISTANCE(v.embedding, :1, COSINE) AS similarity
            FROM   PIPELINE_LOG_VECTORS v
            JOIN   PIPELINE_LOGS l  ON l.log_id = v.id
            JOIN   PIPELINE_JOBS j  ON j.job_id = l.job_id
            {where_sql}
            ORDER  BY similarity
            FETCH  FIRST 10 ROWS ONLY
        """

        cur = connection.cursor()
        cur.execute(sql, bind_params)
        rows = cur.fetchall()

        if not rows:
            return "No matching log entries found."

        output_lines: list[str] = []
        for i, (dag, task, exec_date, status, level, message, sim) in enumerate(rows, 1):
            output_lines.append(
                f"[{i}] sim={sim:.4f} | {level} | {dag}.{task} | {exec_date} | status={status}\n"
                f"    {message}"
            )
        return "\n\n".join(output_lines)
    except Exception as e:
        return f"Hybrid search error: {e}"


tools = [pipeline_sql_tool, log_search_tool, hybrid_pipeline_search]
print(f"\u2705 {len(tools)} tools registered: {', '.join(t.name for t in tools)}")

✅ LLM ready: qwen2.5 @ http://host.docker.internal:11434
✅ 3 tools registered: pipeline_sql_tool, log_search_tool, hybrid_pipeline_search


## Build the Agent

A ReAct agent (Reason + Act) loops through: observe the question, decide which tool
to call and with what input, observe the tool's output, decide whether to call
another tool or answer. `max_iterations=8` allows multi-tool reasoning chains.

`OracleChatMessageHistory` persists conversation context back into Oracle --
the agent remembers answers from earlier questions in the session. This means
the final reliability report (question 6) can reference patterns discovered in
questions 3 and 5 without re-running those queries.

In [6]:
# -- System prompt --
SYSTEM_PROMPT = (
    "You are a data pipeline reliability expert with access to 90 days "
    "of pipeline run history for a NYC Taxi data warehouse. "
    "You have three tools: "
    "(1) pipeline_sql_tool for structured questions -- counts, durations, failure rates, date ranges; "
    "(2) log_search_tool for questions requiring understanding of log text, error messages, "
    "warnings, schema changes, and data quality issues; "
    "(3) hybrid_pipeline_search for questions that combine both -- filter by DAG or date range "
    "AND search log content semantically. "
    "Always consider using multiple tools when a question could benefit from both structured "
    "and semantic perspectives. Be specific: include DAG names, dates, row counts, error codes, "
    "and durations. When you identify patterns, explain them and suggest what to investigate next."
)

# -- LangGraph ReAct agent --
# ChatOllama (not OllamaLLM) is required -- chat models support bind_tools.
agent_executor = create_agent(
    model=llm,
    tools=tools,
)

# -- In-memory chat history --
chat_history = InMemoryChatMessageHistory()


def ask(question: str) -> str:
    """Run one question through the agent, preserving chat history across calls."""
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + list(chat_history.messages) + [HumanMessage(content=question)]
    result = agent_executor.invoke({"messages": messages})
    answer = result["messages"][-1].content
    chat_history.add_user_message(question)
    chat_history.add_ai_message(answer)
    return answer


print("✅ Agent ready (LangGraph ReAct). Chat history active for this session.")


✅ Agent ready (LangGraph ReAct). Chat history active for this session.


## The Demo

Six questions designed to exercise each capability tier in order -- from simple SQL
to full multi-tool reasoning with memory across questions:

| # | Tier | What it exercises |
|---|------|-------------------|
| 1 | Tier 1 | Pure SQL: failure counts and ranking |
| 2 | Tier 2 | Vector search: schema drift in log text only |
| 3 | Tier 3 | Agent reasoning: duration trend across multiple runs |
| 4 | Tier 2 | Vector search: silent issues that didn't cause failures |
| 5 | Tier 3 | Agent reasoning: temporal pattern requiring aggregation |
| 6 | Hybrid | All three tools + memory from questions 1-5 |

Each question is run with `verbose=True`, so you can watch the agent's reasoning
chain -- which tool it picks, what it passes, what it receives.

In [7]:
# Question 1/6  [Tier 1 -- Structured SQL]
print("=" * 70)
print(" Q1/6  [Tier 1 -- Structured SQL]")
print("=" * 70)
print(" How many pipeline failures occurred in the last 30 days, and which DAG failed most often?")
print("-" * 70)
answer = ask("How many pipeline failures occurred in the last 30 days, and which DAG failed most often?")
print()
print(answer)


 Q1/6  [Tier 1 -- Structured SQL]
 How many pipeline failures occurred in the last 30 days, and which DAG failed most often?
----------------------------------------------------------------------
[TOOL] pipeline_sql_tool called with: SELECT dag_id, COUNT(*) AS failure_count FROM PIPELINE_JOBS WHERE status = 'fail
[TOOL] pipeline_sql_tool called with: SELECT dag_id, COUNT(*) AS failure_count FROM PIPELINE_JOBS WHERE status = 'fail

Based on the query results, here are the pipeline failures that occurred in the last 30 days:

- The DAG `borough_partition_load` failed 12 times.
- The DAG `fare_aggregation_daily` failed 8 times.
- The DAG `nyc_taxi_ingestion` failed 4 times.
- The DAG `data_quality_checks` failed 4 times.

The DAG that failed most often in the last 30 days is `borough_partition_load`, with 12 failures.

Next, we might want to investigate why this particular DAG is experiencing more frequent failures. We can use the log_search_tool or hybrid_pipeline_search to look for patt

In [8]:
# Question 2/6  [Tier 2 -- Vector Search (log text)]
print("=" * 70)
print(" Q2/6  [Tier 2 -- Vector Search (log text)]")
print("=" * 70)
print(" Which jobs had schema-related failures? What exactly changed in the schema?")
print("-" * 70)
answer = ask("Which jobs had schema-related failures? What exactly changed in the schema?")
print()
print(answer)


 Q2/6  [Tier 2 -- Vector Search (log text)]
 Which jobs had schema-related failures? What exactly changed in the schema?
----------------------------------------------------------------------
[TOOL] log_search_tool called with: schema change failure

Based on the log entries, there were schema-related failures in two specific tasks:

1. **`data_quality_checks.check_schema_drift`**:
   - Failed multiple times on dates: 2026-02-14, 2026-01-30, 2026-01-24, and 2026-02-28.
   
2. **`nyc_taxi_ingestion.validate_schema`**:
   - Failed on date: 2026-02-07.

To understand the exact changes in the schema that caused these failures, we need to inspect the logs for each of these tasks. Let's use the `hybrid_pipeline_search` tool to get more context and details from the task logs.
Would you like me to proceed with this?


In [9]:
# Question 3/6  [Tier 3 -- Agent Reasoning (cross-run pattern)]
print("=" * 70)
print(" Q3/6  [Tier 3 -- Agent Reasoning (cross-run pattern)]")
print("=" * 70)
print(" Why does the tip_analysis_weekly job keep getting slower over time?")
print("-" * 70)
answer = ask("Why does the tip_analysis_weekly job keep getting slower over time?")
print()
print(answer)


 Q3/6  [Tier 3 -- Agent Reasoning (cross-run pattern)]
 Why does the tip_analysis_weekly job keep getting slower over time?
----------------------------------------------------------------------
[TOOL] pipeline_sql_tool called with: SELECT execution_date, duration_seconds FROM PIPELINE_JOBS WHERE dag_id = 'tip_a
[TOOL] hybrid_pipeline_search called | query=schema change OR data volume increase | dag=tip_analysis_weekly | from=2026-01-01 | to=2026-04-30
[TOOL] log_search_tool called with: schema change OR data quality issue

The log entries indicate that the `check_schema_drift` task in the `data_quality_checks` DAG failed multiple times on specific dates, which aligns with the observed increase in job duration. Here are the relevant logs:

1. **2026-02-14 09:59:45**
   - `ERROR | data_quality_checks.check_schema_drift`
   - Executor exited with code 1.

2. **2026-01-30 09:59:45**
   - `ERROR | data_quality_checks.check_schema_drift`
   - Executor exited with code 1.

3. **2026-01-24 09

In [10]:
# Question 4/6  [Tier 2 -- Vector Search (silent issues)]
print("=" * 70)
print(" Q4/6  [Tier 2 -- Vector Search (silent issues)]")
print("=" * 70)
print(" Are there any silent data quality issues I should know about that did not cause job failures?")
print("-" * 70)
answer = ask("Are there any silent data quality issues I should know about that did not cause job failures?")
print()
print(answer)


 Q4/6  [Tier 2 -- Vector Search (silent issues)]
 Are there any silent data quality issues I should know about that did not cause job failures?
----------------------------------------------------------------------
[TOOL] hybrid_pipeline_search called | query=data quality warning or info | dag= | from=2026-01-01 | to=2026-03-31
[TOOL] log_search_tool called with: data quality warning OR info

The log entries indicate that the `check_fare_ranges` task within the `data_quality_checks` DAG has failed multiple times, which suggests potential silent data quality issues. Here are the relevant logs:

1. **2026-03-16 09:59:45**
   - `ERROR | data_quality_checks.check_fare_ranges`
   - Unexpected failure in check_fare_ranges: executor exited with code 1.

2. **2026-02-25 09:59:45**
   - `ERROR | data_quality_checks.check_fare_ranges`
   - Unexpected failure in check_fare_ranges: executor exited with code 1.

3. **2026-01-11 09:59:45**
   - `ERROR | data_quality_checks.check_fare_ranges`
   - Un

In [11]:
# Question 5/6  [Tier 3 -- Agent Reasoning (temporal pattern)]
print("=" * 70)
print(" Q5/6  [Tier 3 -- Agent Reasoning (temporal pattern)]")
print("=" * 70)
print(" Compare Monday vs other weekdays for pipeline duration. Is there a pattern worth investigating?")
print("-" * 70)
answer = ask("Compare Monday vs other weekdays for pipeline duration. Is there a pattern worth investigating?")
print()
print(answer)


 Q5/6  [Tier 3 -- Agent Reasoning (temporal pattern)]
 Compare Monday vs other weekdays for pipeline duration. Is there a pattern worth investigating?
----------------------------------------------------------------------
[TOOL] pipeline_sql_tool called with: SELECT 
    execution_date,
    CASE 
        WHEN TO_CHAR(execution_date, 'DY')
[TOOL] pipeline_sql_tool called with: SELECT 
    execution_date,
    CASE 
        WHEN TO_CHAR(execution_date, 'DY')
[TOOL] pipeline_sql_tool called with: SELECT 
    execution_date,
    CASE 
        WHEN TO_CHAR(execution_date, 'DY')

Based on the query results, we can observe the following patterns:

### Average Pipeline Durations

- **Monday Averages**:
  - Monday durations range from approximately 183 seconds to 353 seconds.
  - The average duration for Mondays is around 297.64 seconds.

- **Other Weekdays Averages**:
  - Other weekdays' durations range from approximately 167.875 seconds to 241.75 seconds.
  - The average duration for other wee

In [12]:
# Question 6/6  [Hybrid -- All tools + memory from prior questions]
print("=" * 70)
print(" Q6/6  [Hybrid -- All tools + memory from prior questions]")
print("=" * 70)
print(" Give me a full reliability report for the borough_partition_load DAG -- failures, patterns, recommendations.")
print("-" * 70)
answer = ask("Give me a full reliability report for the borough_partition_load DAG -- failures, patterns, recommendations.")
print()
print(answer)


 Q6/6  [Hybrid -- All tools + memory from prior questions]
 Give me a full reliability report for the borough_partition_load DAG -- failures, patterns, recommendations.
----------------------------------------------------------------------
[TOOL] pipeline_sql_tool called with: SELECT execution_date, status, error_code, rows_processed FROM PIPELINE_JOBS WHE
[TOOL] hybrid_pipeline_search called | query=What are the common errors and patterns in borough_partition_load DAG? | dag=borough_partition_load | from= | to=
[TOOL] hybrid_pipeline_search called | query=How can we improve the reliability of borough_partition_load DAG? | dag=borough_partition_load | from= | to=
[TOOL] pipeline_sql_tool called with: SELECT execution_date, status, error_code, rows_processed FROM PIPELINE_JOBS WHE
[TOOL] hybrid_pipeline_search called | query=What are the common errors and patterns in borough_partition_load DAG? | dag=borough_partition_load | from= | to=

Based on the provided responses, we have identifi

## What Was Built

This notebook demonstrated a complete, locally-running pipeline intelligence agent
built on Oracle AI Database. Here is what each component contributed:

### Architecture

| Component | Role |
|-----------|------|
| Oracle AI Database | Single store for relational job metadata, free-text logs, and vector embeddings |
| `PIPELINE_JOBS` + `PIPELINE_LOGS` | Structured + unstructured data in the same schema |
| `PIPELINE_LOG_VECTORS` | Vector index on log text, queryable via `VECTOR_DISTANCE()` |
| `AGENT_MEMORY` | Persistent conversation context via `OracleChatMessageHistory` |
| `nomic-embed-text` | Purpose-built embedding model -- not the chat model |
| `mistral` (or your LLM) | Agent reasoning and natural language response generation |
| LangChain ReAct agent | Orchestration layer -- tool selection, multi-step reasoning |

### Key Capabilities Demonstrated

1. **Converged storage** -- vector embeddings and relational tables in one Oracle instance,
   no separate vector store required

2. **Hybrid query** -- SQL `WHERE` filters + `VECTOR_DISTANCE()` ranking in a single
   statement (`hybrid_pipeline_search` tool)

3. **Dedicated embedding model** -- `nomic-embed-text` produces semantically precise
   vectors for log similarity search, separate from the reasoning LLM

4. **Persistent agent memory** -- `OracleChatMessageHistory` stores session context
   in Oracle, so later questions can reference earlier answers

5. **Tier 2 discovery** -- schema drift, silent data quality issues, and deduplication
   events that have no structured error code were surfaced purely by vector search

6. **Portability** -- swap Ollama for OpenAI by changing three variables in the
   config cell; all LangChain code is identical

### Resources

- [Oracle AI Database 26ai Free Tier](https://www.oracle.com/database/free/)
- [Oracle Developer Resources](https://www.oracle.com/developer/resources/)
- [langchain-oracledb Documentation](https://python.langchain.com/docs/integrations/vectorstores/oracle/)
- [Oracle AI Developer Hub](https://github.com/oracle-devrel/oracle-ai-developer-hub)

---
*Licensed under the Universal Permissive License v 1.0*

In [13]:
results = vector.similarity_search("schema change timestamp column", k=2)
for doc in results:
    print(type(doc))
    print(repr(doc))
    print(vars(doc) if hasattr(doc, '__dict__') else dir(doc))
    print()

NameError: name 'vector' is not defined